# Building a stock database using Polygon
*Requirements: Python 3.11.4, a Polygon.io and a stockanalysis.com subscription*

*Currently undergoing a major overhaul. Not everything may work, especially the updates.*

The goal is to create a financial database of 1-minute OHLCV data from 2004, delisted or listed, for all common stocks (ADR and NYRS included). I use Polygon for price data, which is "ticker-centric". All data from Polygon is point-in-time by ticker. This means we have to take care of renamings ourselves. Some vendors already do renamings (e.g. Tiingo). And almost all of the vendors that do renamings have a small bit of survivorship-bias, because when a ticker is re-used the old data is lost. Polygon does not have this problem.

I will also make sure that it is easy to update without redownloading everything (that is what the 'Update' paragraph in each notebook is for!). However it is *not* optimized to update daily. I only intend to update monthly. My backtesting and live trading framework will be different. E.g. for a trading system that trades daily bars, I will only download the required daily bars every day in the live system and not update the entire database for all tickers. For a day-trading system I will use screeners, which also do not require downloading all 6000+ stocks. In would be very hard to constantly stream 6000+ stocks at the same time, if not impossible.

With a previous project I tried to convert tick quote data to quotes and then merge them with bars to get a realistic bid-ask spread. I have decided to not use quote data anymore. It is simply too expensive, cumbersome or simply impossible. I would have to download all tick data for all dates for all tickers in order to get quotebars, which will be several TBs. Instead, I will just trade using 1-minute OHLC. When backtesting, I will assume a spread of 1 tick or a few cents. I also will avoid illiquid stocks. If I want a more realistic estimate, I will sample the bid-ask spread throughout the day or something. I am not interested in competing with the MMs anyways.

I will organize my data in the following way (only folders and important files shown):

```
├── data_import (code directory)
├── data
│   ├── market
│   │   ├── market_hours.csv
│   │   ├── trading_minutes.csv
│   ├── polygon
│   │   ├── raw
│   │   |   ├── m1
│   │   |   ├── adjustments
│   │   |   ├── tickers
│   │   ├── processed
│   │   |   ├── m1
│   │   |   ├── m5
│   │   |   ├── d1
│   │   ├── secret.txt
│   ├── stockanalysis
│   │   ├── raw
```

The <code>market</code> folder will contain the market hours. All data will be stored in the corresponding directory based on the data vendor. Polygon is for stock data. Stockanalysis is for stock renamings. Raw data is never adjusted. When updating raw data, we should be able to just append it. Processed data is always adjusted which means it is usable by the backtester. When new data comes in, all processed data will be adjusted again. The backtester should only use what is in the <code>processed</code> folders. The <code>secret.txt</code> files contain the API keys. All folders have to be created manually. 

Because the rate limit for the free Polygon plan is quite low, you need at least the starter subscription (29 USD/month). They actually have a 20% discount for students.

The most important file is <code>tickers_v{version}.csv</code> which resides in the <code>data</code> folder. This is a csv containing all tickers. This is the file which you have to loop through when filtering/backtesting. It is not an easy task to build the ticker lists. Or even handle to all other problems such as renamings, delistings, early closes, dividends and splits. I have spent quite some time on this. You *cannot* just loop through the most recent ticker list that Polygon provides.

The series of notebooks:
1. An introduction to the problems with the Polygon ticker list.
2. Get market hours.
3. Solve the ticker problems to create a correct list of tickers.
4. Add ETPs.
5. Download adjustments.
6. Download raw prices.
7. Processing data.
8. Renamings.
9. Aggregate to higher timeframes.
10. Some extra handy functions.

Yes I know that a SQL database is much better, however I care about simplicity more than anything else. It should be straightforward to put everything in a proper database.

*Note: a data point at 15:59 with OHLC means that the open was at 15:59:00 and close at 16:00:00.*

In [92]:
from massive.rest import RESTClient
from datetime import date
import pandas as pd
import mplfinance as mpf
from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

client = RESTClient(api_key=API_KEY)

POLYGON_DATA_PATH = "../data/polygon/"

START_DATE = date(2019, 6, 1)
END_DATE = date(2024, 3, 1)



# 1.1 Tickers
We are only interested in common stocks (CS) and ADR commons (ADRC) in the US only that are not OTC. It must also include the delisted stocks. So we need to get 4 different set of tickers: CS+active, CS+inactive, ADRC+active, ADRC+inactive. Setting the <code>asset_class</code> to "stocks" only pulls non-OTC tickers. The <code>list_tickers</code> returns an iterator. Every iteration is 1 request with the specified <code>limit</code>.


In [93]:
print(pd.DataFrame(client.get_ticker_types(asset_class="stocks", locale="us")))

   asset_class     code                            description locale
0       stocks       CS                           Common Stock     us
1       stocks      PFD                        Preferred Stock     us
2       stocks  WARRANT                                Warrant     us
3       stocks    RIGHT                                 Rights     us
4       stocks     BOND                         Corporate Bond     us
5       stocks      ETF                   Exchange Traded Fund     us
6       stocks      ETN                   Exchange Traded Note     us
7       stocks      ETV                Exchange Traded Vehicle     us
8       stocks       SP                     Structured Product     us
9       stocks     ADRC     American Depository Receipt Common     us
10      stocks     ADRP  American Depository Receipt Preferred     us
11      stocks     ADRW   American Depository Receipt Warrants     us
12      stocks     ADRR     American Depository Receipt Rights     us
13      stocks     F

In [94]:
ticker_list_iterator_active = client.list_tickers(type="CS", date=END_DATE.isoformat(), active=True, market='stocks', limit=1000)
ticker_list_iterator_delisted = client.list_tickers(type="CS", date=END_DATE.isoformat(), active=False, market='stocks', limit=1000)
ticker_list_iterator_active_adr = client.list_tickers(type="ADRC", date=END_DATE.isoformat(), active=True, market='stocks', limit=1000)
ticker_list_iterator_delisted_adr = client.list_tickers(type="ADRC", date=END_DATE.isoformat(), active=False, market='stocks', limit=1000)
tickers_active = pd.DataFrame(ticker_list_iterator_active)
tickers_delisted = pd.DataFrame(ticker_list_iterator_delisted)
tickers_active_adr = pd.DataFrame(ticker_list_iterator_active_adr)
tickers_delisted_adr = pd.DataFrame(ticker_list_iterator_delisted_adr)

In [95]:
tickers_active.head(3)

,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,True,0001090872,BBG000C2V3D6,usd,None,None,None,None,2024-12-03T20:08:18.55486Z,us,stocks,Agilent Technologies Inc.,XNYS,BBG001SCTQY4,A,CS,None
1,True,0001675149,BBG00B3T3HD3,usd,None,None,None,None,2024-12-03T20:08:18.553135Z,us,stocks,Alcoa Corporation,XNYS,BBG00B3T3HF1,AA,CS,None
2,True,0001844817,BBG011XR7306,usd,None,None,None,None,2024-12-03T20:08:18.553533Z,us,stocks,Armada Acquisition Corp. I Common Stock,XNAS,BBG011XR7315,AACI,CS,None


In [96]:
tickers_delisted.head(3)

,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,False,0001011006,BBG000KB2D74,usd,None,None,None,2019-10-07T04:00:00Z,2024-12-03T21:33:22.821052Z,us,stocks,Altaba Inc. Common Stock,XNAS,BBG001S8V781,AABA,CS,None
1,False,0001829432,NaN,usd,None,None,None,2023-11-07T05:00:00Z,2025-08-13T14:20:43.779961Z,us,stocks,Ares Acquisition Corporation,XNYS,NaN,AAC,CS,None
2,False,0001802457,NaN,usd,None,None,None,2021-06-25T04:00:00Z,2024-12-03T22:22:13.502751Z,us,stocks,Artius Acquisition Inc. Class A Common Stock,XNAS,NaN,AACQ,CS,None


In [97]:
tickers_active_adr.head(3)

,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,True,0001420529,BBG000V2S3P6,usd,None,None,None,None,2024-12-03T20:08:18.552947Z,us,stocks,ATA Creativity Global American Depositary Shares,XNAS,BBG001T125S9,AACG,ADRC,None
1,True,0001565025,BBG000BN5VZ4,usd,None,None,None,None,2024-12-03T20:08:18.554563Z,us,stocks,AMBEV S.A.,XNYS,BBG005KLVT74,ABEV,ADRC,None
2,True,0001956827,BBG01JLQNZ03,usd,None,None,None,None,2024-12-03T20:08:18.553525Z,us,stocks,Abivax SA American Depositary Shares,XNAS,BBG01JLQNZV9,ABVX,ADRC,None


In [98]:
tickers_delisted_adr.head(3)

,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,False,0001611787,BBG00K6FMBQ8,usd,None,None,None,2018-02-12T05:00:00Z,2024-12-03T20:51:58.376135Z,us,stocks,Advanced Accelerator Applications S.A. America...,XNAS,BBG007K5CVB6,AAAP,ADRC,None
1,False,0001091587,BBG000DK5Q25,usd,None,None,None,2023-05-23T04:00:00Z,2024-12-03T20:04:49.838299Z,us,stocks,ABB Ltd.,XNYS,BBG001SDDMX9,ABB,ADRC,None
2,False,0001492074,BBG00XRJJLG2,usd,None,None,None,2023-12-07T05:00:00Z,2024-12-03T20:07:15.058378Z,us,stocks,Abcam plc American Depositary Shares,XNAS,BBG00XRJJM98,ABCM,ADRC,None


In [99]:
tickers = pd.concat([tickers_active, tickers_delisted, tickers_active_adr, tickers_delisted_adr]).reset_index()

In [100]:
tickers.head(3)

,index,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,0,True,0001090872,BBG000C2V3D6,usd,None,None,None,None,2024-12-03T20:08:18.55486Z,us,stocks,Agilent Technologies Inc.,XNYS,BBG001SCTQY4,A,CS,None
1,1,True,0001675149,BBG00B3T3HD3,usd,None,None,None,None,2024-12-03T20:08:18.553135Z,us,stocks,Alcoa Corporation,XNYS,BBG00B3T3HF1,AA,CS,None
2,2,True,0001844817,BBG011XR7306,usd,None,None,None,None,2024-12-03T20:08:18.553533Z,us,stocks,Armada Acquisition Corp. I Common Stock,XNAS,BBG011XR7315,AACI,CS,None


Some basic checks to check for weird things:

In [101]:
# Check na values
tickers.isna().sum()

index                       0
active                      0
cik                       336
composite_figi           4338
currency_name               0
currency_symbol         11327
base_currency_symbol    11327
base_currency_name      11327
delisted_utc             5804
last_updated_utc            0
locale                      0
market                      0
name                        0
primary_exchange            0
share_class_figi         4338
ticker                      0
type                        0
source_feed             11327
dtype: int64

In [102]:
# Check unique values 
print(tickers['currency_name'].unique()) # Should be USD only
print(tickers['locale'].unique()) # Should be US only

<StringArray>
['usd']
Length: 1, dtype: str
<StringArray>
['us']
Length: 1, dtype: str


In [103]:
# Clean up and rearrange columns
tickers_active = tickers_active[['ticker', 'name', 'active', 'delisted_utc', 'last_updated_utc', 'cik', 'composite_figi', 'type']]
tickers_delisted = tickers_delisted[['ticker', 'name', 'active', 'delisted_utc', 'last_updated_utc', 'cik', 'composite_figi', 'type']]
tickers_active_adr = tickers_active_adr[['ticker', 'name', 'active', 'delisted_utc', 'last_updated_utc', 'cik', 'composite_figi', 'type']]
tickers_delisted_adr = tickers_delisted_adr[['ticker', 'name', 'active', 'delisted_utc', 'last_updated_utc', 'cik', 'composite_figi', 'type']]
tickers = tickers[['ticker', 'name', 'active', 'delisted_utc', 'last_updated_utc', 'cik', 'composite_figi', 'type']]
tickers.head(3)

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
0,A,Agilent Technologies Inc.,True,None,2024-12-03T20:08:18.55486Z,0001090872,BBG000C2V3D6,CS
1,AA,Alcoa Corporation,True,None,2024-12-03T20:08:18.553135Z,0001675149,BBG00B3T3HD3,CS
2,AACI,Armada Acquisition Corp. I Common Stock,True,None,2024-12-03T20:08:18.553533Z,0001844817,BBG011XR7306,CS


# 1.2 Problem: duplicates (Fixed at Polygon/Massive! See below)

Check if all tickers are unique:

In [104]:
ticker_frequency = tickers.pivot_table(columns=['ticker'], aggfunc='size')
ticker_frequency[ticker_frequency > 1]

Series([], dtype: int64)

In [105]:
ticker_frequency[ticker_frequency > 0]

ticker
A       1
AA      1
AAAP    1
AABA    1
AAC     1
       ..
ZY      1
ZYME    1
ZYNE    1
ZYXI    1
ZZ      1
Length: 11327, dtype: int64

OUTDATED: Apparently they are not unique. There are more than 1800 duplications. Maybe it is because tickers get recycled after delisting?

Polygon/Massive fixed this! No more duplicates! See below.

And just look at examples in the old notebook and now: AA, ZME

In [106]:
# Assuming tickers is a DataFrame with a 'ticker' column
ticker_frequency = tickers['ticker'].value_counts()
ticker_frequency[ticker_frequency > 1]

Series([], Name: count, dtype: int64)

In [107]:
ticker_frequency[ticker_frequency > 0]

ticker
A       1
AA      1
AACI    1
AACT    1
AADI    1
       ..
ZEAL    1
ZME     1
ZNH     1
ZPIN    1
ZX      1
Name: count, Length: 11327, dtype: int64

In [108]:
# 1. Check the total number of rows in your ticker column
print(f"Total ticker entries: {len(tickers['ticker'])}")

# 2. Show the full frequency distribution (including counts of 1)
ticker_frequency = tickers['ticker'].value_counts()
display(ticker_frequency)   # This will show all counts, maybe with many rows

# 3. Check the maximum frequency
max_freq = ticker_frequency.max()
print(f"Maximum ticker frequency: {max_freq}")

# 4. See how many unique tickers exist
print(f"Number of unique tickers: {ticker_frequency.shape[0]}")

Total ticker entries: 11327


ticker
A       1
AA      1
AACI    1
AACT    1
AADI    1
       ..
ZEAL    1
ZME     1
ZNH     1
ZPIN    1
ZX      1
Name: count, Length: 11327, dtype: int64

Maximum ticker frequency: 1
Number of unique tickers: 11327


In [109]:
ticker_frequency_active = tickers_active.pivot_table(columns=['ticker'], aggfunc='size')
ticker_frequency_active[ticker_frequency_active > 1]

Series([], dtype: int64)

There are indeed no duplicates in the active stocks. The tickers are recycled:

In [110]:
tickers[tickers['ticker'] == "AAN"]


,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
9,AAN,"The Aaron's Company, Inc.",True,None,2024-12-03T20:08:18.552904Z,0001821393,BBG00WCNDCZ6,CS


OLD/OUTDATED:

However this is not even a real recycling. It's the *same* company. Yet it has different entries because it changed its name. We will have to merge them later.

We need to keep this in mind if we download historical data. Because we cannot just loop through all tickers and then store them in a csv. Because then the first ones will be overwritten.

The cik is also not unique.

In [111]:
ticker_frequency = tickers.pivot_table(columns=['cik'], aggfunc='size')
ticker_frequency[ticker_frequency > 1]

cik
0000001800    2
0000002186    2
0000003197    2
0000003453    2
0000004457    2
             ..
0001968487    2
0001968915    2
0001971543    2
0001974138    2
0001978867    2
Length: 1400, dtype: int64

In [112]:
tickers[tickers['cik'] == "0000002186"]

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
688,BKTI,BK Technologies Corporation,True,None,2024-12-03T20:08:18.55437Z,0000002186,BBG00NKSM4N7,CS
9484,RWC,RELM Wireless Corporation,False,2018-06-05T04:00:00Z,2024-12-03T20:52:46.875723Z,0000002186,NaN,CS


In [113]:
tickers[tickers['ticker'] == "FB"]


,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type


In [114]:
tickers[tickers['ticker'] == "META"]


,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
3032,META,"Meta Platforms, Inc. Class A Common Stock",True,None,2024-12-03T20:08:18.553418Z,0001326801,BBG000MM2P62,CS


If a ticker is *actually* delisted we can handle this in at least 2 different ways:
* The [Norgate](https://norgatedata.com/data-package-faq.php) way: the data of active tickers is just saved under the ticker name. For delisted tickers the delisting date is appended to the ticker. E.g. FB-2022-06.csv
* My preferred way: Always assign an unique ID to the stocks. I chose to always append the *listing date* to the stock. This is because the listing date never changes. I am aware that other identifiers such as CUSIP exists. However Polygon does not even provide this and everyone uses another identifier. I like to be 'ticker-centric' because that coincides with charting platforms. I want to keep a history of all ticker changes to always be able to map it to other identifiers.

We need to keep in mind that it is possible for a ticker to go to OTC and then back (see HTZ on TradingView). We will treat these cases as two different stocks, such that we do not trade OTC. We won't even use the OTC data.

In [ ]:
tickers['delisted_utc']

0                        None
1                        None
11093    2018-02-12T05:00:00Z
5315     2019-10-07T04:00:00Z
5316     2023-11-07T05:00:00Z
                 ...         
10673    2022-10-20T04:00:00Z
5313                     None
10674    2023-10-12T04:00:00Z
5314                     None
10675    2013-03-19T04:00:00Z
Name: delisted_utc, Length: 11327, dtype: object

In [ ]:
# replaced with proper datetime64 values, making it easier to perform date-based operations (e.g., filtering, resampling, extracting components).
tickers['delisted_utc'] = pd.to_datetime(tickers['delisted_utc'])
tickers['delisted_utc']

0                             NaT
1                             NaT
11093   2018-02-12 05:00:00+00:00
5315    2019-10-07 04:00:00+00:00
5316    2023-11-07 05:00:00+00:00
                   ...           
10673   2022-10-20 04:00:00+00:00
5313                          NaT
10674   2023-10-12 04:00:00+00:00
5314                          NaT
10675   2013-03-19 04:00:00+00:00
Name: delisted_utc, Length: 11327, dtype: datetime64[us, UTC]

In [122]:
tickers.sort_values(by=['ticker'], inplace=True)
# We are not interested in stocks that were delisted before the start date
tickers['delisted_utc'] = pd.to_datetime(tickers['delisted_utc'])
tickers['last_updated_utc'] = pd.to_datetime(tickers['last_updated_utc'])
tickers = tickers[(tickers['delisted_utc'].dt.date > START_DATE) | tickers['delisted_utc'].isnull()]
tickers.reset_index(inplace=True, drop=True)
tickers

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
0,A,Agilent Technologies Inc.,True,NaT,2024-12-03 20:08:18.554860+00:00,0001090872,BBG000C2V3D6,CS
1,AA,Alcoa Corporation,True,NaT,2024-12-03 20:08:18.553135+00:00,0001675149,BBG00B3T3HD3,CS
2,AABA,Altaba Inc. Common Stock,False,2019-10-07 04:00:00+00:00,2024-12-03 21:33:22.821052+00:00,0001011006,BBG000KB2D74,CS
3,AAC,Ares Acquisition Corporation,False,2023-11-07 05:00:00+00:00,2025-08-13 14:20:43.779961+00:00,0001829432,NaN,CS
4,AACG,ATA Creativity Global American Depositary Shares,True,NaT,2024-12-03 20:08:18.552947+00:00,0001420529,BBG000V2S3P6,ADRC
...,...,...,...,...,...,...,...,...
8454,ZWS,Zurn Elkay Water Solutions Corporation,True,NaT,2024-12-03 20:08:18.554557+00:00,0001439288,BBG000H8R0N8,CS
8455,ZY,Zymergen Inc. Common Stock,False,2022-10-20 04:00:00+00:00,2024-12-03 22:29:24.707484+00:00,0001645842,BBG0077HPN74,CS
8456,ZYME,Zymeworks Inc.,True,NaT,2024-12-03 20:08:18.553445+00:00,0001937653,BBG019XSYC89,CS
8457,ZYNE,"Zynerba Pharmaceuticals, Inc",False,2023-10-12 04:00:00+00:00,2024-12-03 20:06:34.963121+00:00,0001621443,BBG007BBS8B7,CS


# 1.3 Problem: no start dates
We now have the end dates. But to download the data we also need the start date. Let's try to get the start date of META. To get them we could use the <code>Ticker Details</code> endpoint. But this gets the *list date*. This is not necessarily the same as the start date of a ticker. E.g. FB changed their ticker to META, but what does ticker details give? We want 2022.

In [118]:
meta_details = client.get_ticker_details(ticker = "META")
meta_details.list_date

'2012-05-18'

In [119]:
result = client.get_ticker_events(ticker="META")
result

TickerChangeResults(name='Meta Platforms, Inc. Class A Common Stock', composite_figi='BBG000MM2P62', cik='0001326801', events=[{'ticker_change': {'ticker': 'META'}, 'type': 'ticker_change', 'date': '2022-06-09'}, {'ticker_change': {'ticker': 'FB'}, 'type': 'ticker_change', 'date': '2012-05-18'}])

The start date of META is 2022-06-09. But the list date is the one from FB from 2012. 

What happens if we download data from a ticker that had ticker changes, without the correct start/end date?

In [120]:
meta_iterator = client.get_aggs(ticker = "META", multiplier = 1, timespan = "day", from_ = "2010-01-01", to = "2023-08-18", limit=50000)
meta_d1 = pd.DataFrame(meta_iterator)
meta_d1["timestamp"] = pd.to_datetime(meta_d1["timestamp"], unit="ms").dt.date
meta_d1.rename(columns={"timestamp": "date"}, inplace=True)
meta_d1

,open,high,low,close,volume,vwap,date,transactions,otc
0,15.0800,15.19,15.042,15.12,334935.0,15.1107,2021-06-30,2101,None
1,15.1300,15.13,14.840,14.89,241629.0,15.0126,2021-07-01,2092,None
2,15.0200,15.10,14.930,15.00,388152.0,15.0224,2021-07-02,4733,None
3,15.0800,15.09,14.870,15.01,685094.0,15.1119,2021-07-06,11269,None
4,15.0601,15.15,14.840,14.89,362538.0,14.9830,2021-07-07,5538,None
...,...,...,...,...,...,...,...,...,...
443,300.9800,306.21,298.250,306.19,15641921.0,303.0529,2023-08-14,206053,None
444,306.1400,307.23,300.030,301.95,11623613.0,302.9223,2023-08-15,171652,None
445,300.1950,301.08,294.280,294.29,18547741.0,297.3013,2023-08-16,261408,None
446,293.0500,296.05,284.950,285.09,23950089.0,289.9367,2023-08-17,311835,None


In [121]:
meta_d1.set_index("date", inplace=True)
meta_d1.index = pd.to_datetime(meta_d1.index)
mpf.plot(meta_d1, type='ohlc', show_nontrading=True)

AttributeError: module 'pandas' has no attribute 'core'

So if a ticker was used by another company/ETF and when you download data from the ticker, you get them *both*. The first ticker that used META was actually a metaverse ETF. However what we need is the data from FB up to the renaming, and then the META data.

So to reiterate, you cannot just get the current ticker list from Polygon and download all data. Else you will miss the old facebook data AND get wrong data for META.

In [ ]:
fb_iterator = client.get_aggs(ticker = "FB", multiplier = 1, timespan = "day", from_ = "2010-01-01", to = "2023-08-18", limit=50000)
fb_d1 = pd.DataFrame(fb_iterator)
fb_d1["timestamp"] = pd.to_datetime(fb_d1["timestamp"], unit="ms").dt.date
fb_d1.rename(columns={"timestamp": "date"}, inplace=True)
fb_d1

,open,high,low,close,volume,vwap,date,transactions,otc
0,167.83,168.900,167.2789,168.70,10381490.0,168.2960,2019-04-01,80806,None
1,170.14,174.900,169.5500,174.20,23846529.0,173.2804,2019-04-02,164103,None
2,174.50,177.960,172.9500,173.54,27590058.0,175.1656,2019-04-03,192745,None
3,176.02,178.000,175.5301,176.02,17780675.0,176.7681,2019-04-04,132907,None
4,176.88,177.000,175.1000,175.72,9594133.0,175.7471,2019-04-05,75071,None
...,...,...,...,...,...,...,...,...,...
800,188.45,200.935,187.7300,198.86,31951582.0,195.8094,2022-06-02,336899,None
801,195.98,196.610,189.7800,190.78,19464993.0,192.0885,2022-06-03,239486,None
802,193.99,196.920,188.4000,194.25,30574242.0,193.2857,2022-06-06,307707,None
803,191.93,196.530,191.4900,195.65,18628687.0,194.4583,2022-06-07,204288,None


**The FB ticker also does not show up in the ticker list!** Not even with a <code>active</code> set to False.

In [ ]:
tickers[tickers['ticker'] == "FB"]

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type


It *does* show up in the Polygon ticker list from before the renaming.

In [ ]:
# Before the renaming on 6-9
ticker_list_iterator_fb = client.list_tickers(ticker="FB", type="CS", date="2022-06-08", active=True, market='stocks', limit=1000)
df = pd.DataFrame(ticker_list_iterator_fb)
df

,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,True,0001326801,BBG000MM2P62,usd,None,None,None,None,2022-06-08T00:00:00Z,us,stocks,"Meta Platforms, Inc. Class A Common Stock",XNAS,BBG001SQCQC5,FB,CS,None


In [ ]:
# After the renaming on 6-9: No FB but we do have META
ticker_list_iterator_fb = client.list_tickers(ticker="FB", type="CS", date="2022-06-09", active=True, market='stocks', limit=1000)
df = pd.DataFrame(ticker_list_iterator_fb)
df

""


In [ ]:
ticker_list_iterator_fb = client.list_tickers(ticker="META", type="CS", date="2022-06-09", active=True, market='stocks', limit=1000)
df = pd.DataFrame(ticker_list_iterator_fb)
df

,active,cik,composite_figi,currency_name,currency_symbol,base_currency_symbol,base_currency_name,delisted_utc,last_updated_utc,locale,market,name,primary_exchange,share_class_figi,ticker,type,source_feed
0,True,0001326801,BBG000MM2P62,usd,None,None,None,None,2022-06-09T00:00:00Z,us,stocks,"Meta Platforms, Inc. Class A Common Stock",XNAS,BBG001SQCQC5,META,CS,None


Apparently when a stock renames, the <code>Tickers</code> gets updated by **removing** the old ticker and adding the new one. It also does not treat it as a delisting (there is no <code>delisted_utc</code>). 

This is quite problematic. The only way to get the end date of FB is to loop through all trading days in <code>Tickers</code> and see when the ticker dissappears. The only way to get the start date of the META or FB is also to loop, but then to track when the ticker appears. We cannot use the <code>Ticker Events</code> because it does not include delisted stocks. So information actually gets lost in the <code>Tickers</code> endpoint. Although you would just expect the <code>active</code> flag to be set to False.

Who thought that just getting all available tickers would be such a headache? We haven't even begun downloading data.

To summarize:
- To download the data, we need precise start and end dates. If we use wrong start dates, price data from two different companies but same ticker can be merged (META + META ETF).
    - The <code>list_ticker</code> function does not provide start dates. 
- Downloading data is 'point in time': if a ticker is renamed you still need the old ticker to get the old data. So we also need the old tickers.
    - If a ticker is renamed, the <code>list_ticker</code> function with the current date loses the ticker data, unless you use the old dates. E.g. FB. 

So we will loop through the ticker list for ALL days and create an own version of a list of all tickers with start and end dates. 

If a ticker changes, we thus treat them as 2 different stocks. In the end, we should thus get a META.csv and a FB.csv. We can merge them later *after* downloading all data (there is no other way, as you must use the FB ticker to get the old META data). 

We will take a short stop to retrieve the market hours. This is important if we want to loop through the trading days to retrieve the Polygon ticker list for each day.

# 1.4 Problem: incorrectly labeled stocks

Instead of 'CS', many stocks in the old ticker lists are labeled 'None'. I only found this out later when I downloaded the entire history.

In [ ]:
ticker_list_iterator = client.list_tickers(date="2015-09-20", active=True, market='stocks', limit=1000)
df = pd.DataFrame(ticker_list_iterator)

In [ ]:
df[df['ticker'] == 'AAPL'][['ticker', 'type']]

,ticker,type
12,AAPL,None
